### **ROBUSTEZZA DEI MODELLI FINALI**

Ristima dei modelli finali del notebook 03 sui dati principali e sul campione filtrato con criterio IQR. Tutte le stime usano errori standard robusti HC3.

### **IMPORTO LIBRERIE**

In [ ]:
from pathlib import Path

import pandas as pd
import statsmodels.formula.api as smf

### **IMPORTO DATI E POOLING**

In [ ]:
processed_dir = Path("datas/processed")

def carica_pool(nome_software, nome_hardware):
    software_df = pd.read_csv(processed_dir / nome_software).assign(software=1)
    hardware_df = pd.read_csv(processed_dir / nome_hardware).assign(software=0)
    return pd.concat([software_df, hardware_df], ignore_index=True)

df_principale = carica_pool("dataset_software.csv", "dataset_hardware.csv")
df_iqr = carica_pool("dataset_software_test.csv", "dataset_hardware_test.csv")

print(f"Osservazioni pooled principali: {len(df_principale)}")
print(f"Osservazioni pooled IQR: {len(df_iqr)}")

### **SPECIFICHE E STIMA DEI MODELLI**

In [ ]:
regressori_base = [
    "debt_equity_medio",
    "liquidita_media",
    "log_attivo_medio",
    "log_eta_azienda",
]

formula_roe_pooled = (
    "roe_medio ~ debt_equity_medio + liquidita_media + log_attivo_medio "
    "+ log_eta_azienda + software"
)

formula_roa_interazioni = (
    "roa_medio ~ "
    + " + ".join(regressori_base + ["software"])
    + " + "
    + " + ".join(f"software:{regressore}" for regressore in regressori_base)
)

variabili_roe = ["roe_medio", *regressori_base, "software"]
variabili_roa = ["roa_medio", *regressori_base, "software"]

def stima_hc3(df, variabili, formula):
    dati_modello = df[variabili].dropna().copy()
    modello = smf.ols(formula=formula, data=dati_modello).fit(cov_type="HC3")
    return modello, dati_modello

modello_roe_principale, dati_roe_principale = stima_hc3(
    df_principale,
    variabili_roe,
    formula_roe_pooled,
)
modello_roe_iqr, dati_roe_iqr = stima_hc3(
    df_iqr,
    variabili_roe,
    formula_roe_pooled,
)
modello_roa_principale, dati_roa_principale = stima_hc3(
    df_principale,
    variabili_roa,
    formula_roa_interazioni,
)
modello_roa_iqr, dati_roa_iqr = stima_hc3(
    df_iqr,
    variabili_roa,
    formula_roa_interazioni,
)

modelli = {
    "ROE-principale": (modello_roe_principale, dati_roe_principale),
    "ROE-IQR": (modello_roe_iqr, dati_roe_iqr),
    "ROA-principale": (modello_roa_principale, dati_roa_principale),
    "ROA-IQR": (modello_roa_iqr, dati_roa_iqr),
}

### **REGRESSIONI IQR**

Stima dei due modelli finali sul campione filtrato con criterio IQR ed esportazione delle rispettive tabelle coefficienti in LaTeX.

In [ ]:
export_dir = Path("robustezza_export/tex")
export_dir.mkdir(parents=True, exist_ok=True)

def costruisci_tabella_coefficienti(modello):
    conf_int = modello.conf_int()
    return pd.DataFrame(
        {
            "Coefficiente": modello.params,
            "Errore standard HC3": modello.bse,
            "t": modello.tvalues,
            "p-value": modello.pvalues,
            "IC 95% inf.": conf_int[0],
            "IC 95% sup.": conf_int[1],
        }
    )

tabella_roe_iqr = costruisci_tabella_coefficienti(modello_roe_iqr)
tabella_roa_iqr = costruisci_tabella_coefficienti(modello_roa_iqr)

latex_roe_iqr_path = export_dir / "modello_roe_iqr.tex"
latex_roa_iqr_path = export_dir / "modello_roa_iqr.tex"

tabella_roe_iqr.to_latex(
    latex_roe_iqr_path,
    float_format="%.4f",
    caption="Coefficienti del modello ROE-IQR con errori standard robusti HC3",
    label="tab:modello_roe_iqr",
)
tabella_roa_iqr.to_latex(
    latex_roa_iqr_path,
    float_format="%.4f",
    caption="Coefficienti del modello ROA-IQR con errori standard robusti HC3",
    label="tab:modello_roa_iqr",
)

print("--- Modello ROE-IQR ---")
print(modello_roe_iqr.summary())
print(f"\nTabella ROE-IQR esportata in: {latex_roe_iqr_path}")
print("\n--- Modello ROA-IQR ---")
print(modello_roa_iqr.summary())
print(f"\nTabella ROA-IQR esportata in: {latex_roa_iqr_path}")

display(tabella_roe_iqr)
display(tabella_roa_iqr)

### **TABELLA DI CONFRONTO**

Ogni cella riporta coefficiente e p-value robusto HC3. Asterischi: `***` p<0.01, `**` p<0.05, `*` p<0.10.

In [ ]:
ordine_parametri = [
    "Intercept",
    "debt_equity_medio",
    "liquidita_media",
    "log_attivo_medio",
    "log_eta_azienda",
    "software",
    "software:debt_equity_medio",
    "software:liquidita_media",
    "software:log_attivo_medio",
    "software:log_eta_azienda",
]

etichette_parametri = {
    "Intercept": "Intercetta",
    "debt_equity_medio": "Debt/Equity medio",
    "liquidita_media": "Liquidita media",
    "log_attivo_medio": "Log attivo medio",
    "log_eta_azienda": "Log eta azienda",
    "software": "Software",
    "software:debt_equity_medio": "Software x Debt/Equity medio",
    "software:liquidita_media": "Software x Liquidita media",
    "software:log_attivo_medio": "Software x Log attivo medio",
    "software:log_eta_azienda": "Software x Log eta azienda",
}

def asterischi(p_value):
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""

def formatta_stima(modello, parametro):
    if parametro not in modello.params.index:
        return ""
    coefficiente = modello.params[parametro]
    p_value = modello.pvalues[parametro]
    return f"{coefficiente:.4f}{asterischi(p_value)}\np-value={p_value:.4f}"

tabella_confronto = pd.DataFrame(
    index=[etichette_parametri[parametro] for parametro in ordine_parametri],
    columns=modelli.keys(),
)

for nome, (modello, _) in modelli.items():
    for parametro in ordine_parametri:
        tabella_confronto.loc[etichette_parametri[parametro], nome] = formatta_stima(
            modello,
            parametro,
        )

tabella_confronto.loc["N"] = [
    f"{len(dati_modello)}" for _, dati_modello in modelli.values()
]
tabella_confronto.loc["R2 corretto"] = [
    f"{modello.rsquared_adj:.4f}" for modello, _ in modelli.values()
]

display(
    tabella_confronto.style
    .set_caption("Confronto modelli finali - HC3")
    .set_properties(**{"white-space": "pre-line"})
)

### **ESPORTAZIONE TABELLA DI CONFRONTO IN LATEX**

In [ ]:
export_dir = Path("robustezza_export/tex")
export_dir.mkdir(parents=True, exist_ok=True)

tabella_confronto_latex = tabella_confronto.copy()
for colonna in tabella_confronto_latex.columns:
    tabella_confronto_latex[colonna] = tabella_confronto_latex[colonna].map(
        lambda valore: valore.replace("\n", " ") if isinstance(valore, str) else valore
    )

latex_confronto_path = export_dir / "tabella_confronto_robustezza.tex"
tabella_confronto_latex.to_latex(
    latex_confronto_path,
    escape=False,
    caption="Confronto dei modelli finali principali e IQR con errori standard robusti HC3",
    label="tab:confronto_robustezza",
)

print(f"Tabella di confronto esportata in: {latex_confronto_path}")